# Atenciones de urgencia 2024: un histograma por columna

Fuente: DEIS-MINSAL (`AtencionesUrgencia2024.zip`, 8,97 millones de filas, 21 columnas).

- Columnas **numéricas** (`Total`, tramos de edad, `semana`): histograma.
- Columnas **de texto o código** (establecimiento, causa, región, comuna, etc.): barras con las categorías más frecuentes, porque un histograma no tiene sentido ahí.
- `fecha`: histograma por día.

In [ ]:
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

ZIP = "AtencionesUrgencia2024.zip"
CSV = "AtencionesUrgencia2024.csv"

NUM = ["Total", "Menores_1", "De_1_a_4", "De_5_a_14", "De_15_a_64", "De_65_y_mas", "semana"]
CAT = ["IdEstablecimiento", "NEstablecimiento", "IdCausa", "GlosaCausa",
       "GLOSATIPOESTABLECIMIENTO", "GLOSATIPOATENCION", "GlosaTipoCampana",
       "CodigoRegion", "NombreRegion", "CodigoDependencia", "NombreDependencia",
       "CodigoComuna", "NombreComuna"]

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
miles = mtick.FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", "."))

## 1. Carga

Se lee directo del zip (sin descomprimir) y las columnas de texto se guardan como `category` para no llenar la memoria.

In [ ]:
dtypes = {c: "category" for c in CAT}
dtypes.update({c: "int32" for c in NUM})

with zipfile.ZipFile(ZIP) as z, z.open(CSV) as f:
    df = pd.read_csv(f, sep=";", encoding="latin1", dtype=dtypes, usecols=NUM + CAT + ["fecha"])

df["fecha"] = pd.to_datetime(df["fecha"], format="%d/%m/%Y")
print(f"{len(df):,} filas x {df.shape[1]} columnas | {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
df.head()

## 2. Columnas numéricas

La mayoría de las filas tiene conteos bajos o cero y unas pocas tienen valores muy altos, así que el eje Y va en escala logarítmica. Si no, solo se vería la primera barra.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 13))
for ax, col in zip(axes.flat, NUM):
    serie = df[col]
    if col == "semana":
        ax.hist(serie, bins=range(serie.min(), serie.max() + 2), align="left", color="#2a78d6")
        ax.set_xlabel("semana del año")
    else:
        ax.hist(serie, bins=60, color="#2a78d6")
        ax.set_yscale("log")
        ax.set_xlabel(f"{col} (atenciones por fila)")
    ax.set_title(f"{col}  |  media {serie.mean():.1f}, máx. {serie.max():,}".replace(",", "."), fontsize=10)
    ax.set_ylabel("filas")
    if col != "semana":
        ax.xaxis.set_major_formatter(miles)
for ax in axes.flat[len(NUM):]:
    ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
# Mismos conteos pero sin el 0 y recortados al percentil 99, para ver la forma de la parte baja
fig, axes = plt.subplots(4, 2, figsize=(12, 13))
for ax, col in zip(axes.flat, NUM[:-1]):
    serie = df.loc[df[col] > 0, col]
    tope = serie.quantile(0.99)
    ax.hist(serie[serie <= tope], bins=min(60, int(tope)), color="#2a78d6")
    ax.set_title(f"{col} (> 0, hasta p99 = {tope:.0f})", fontsize=10)
    ax.set_ylabel("filas")
    ax.yaxis.set_major_formatter(miles)
for ax in axes.flat[len(NUM) - 1:]:
    ax.axis("off")
fig.tight_layout()
plt.show()

## 3. Columnas de texto y códigos

Barras horizontales con las 15 categorías más frecuentes (número de filas). El título indica cuántas categorías distintas tiene la columna.

In [ ]:
TOP = 15
fig, axes = plt.subplots(7, 2, figsize=(14, 30))
for ax, col in zip(axes.flat, CAT):
    vc = df[col].value_counts()
    top = vc.head(TOP)[::-1]
    etiquetas = [str(x)[:45] for x in top.index]
    ax.barh(etiquetas, top.values, color="#2a78d6")
    ax.set_title(f"{col}  |  {len(vc):,} categorías".replace(",", "."), fontsize=10)
    ax.xaxis.set_major_formatter(miles)
    ax.set_xlabel("filas")
    ax.tick_params(axis="y", labelsize=8)
for ax in axes.flat[len(CAT):]:
    ax.axis("off")
fig.tight_layout()
plt.show()

## 4. Fecha

Filas por día. Cada día debería tener un número parecido de registros; los huecos o picos son datos faltantes o cambios en la cantidad de establecimientos que reportan.

In [ ]:
por_dia = df["fecha"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.hist(df["fecha"], bins=len(por_dia), color="#2a78d6")
ax.set_title(f"fecha  |  {df['fecha'].min():%d-%m-%Y} a {df['fecha'].max():%d-%m-%Y}, {len(por_dia)} días distintos", fontsize=10)
ax.set_ylabel("filas")
ax.yaxis.set_major_formatter(miles)
fig.tight_layout()
plt.show()